### Task T-01: Load Compatible Models for SAE Analysis on CLIP

This task accomplishes:
1. Loading a pre-trained CLIP model (both vision and text encoders)
2. Setting up hooks to extract intermediate layer activations
3. Verifying the model works with sample inputs (image + text)
4. Inspecting activation shapes to plan SAE training
5. Loading a pre-trained SAE if available|


In [1]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from collections import OrderedDict

# Install dependencies
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install("open_clip_torch")
install("transformers")

import open_clip
from open_clip import create_model_and_transforms, get_tokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00


In [2]:
# Configurations
class Config:
    """Central configuration for model loading and activation extraction."""
    
    # Options: 'ViT-B-32', 'ViT-B-16', 'ViT-L-14'
    MODEL_NAME = 'ViT-B-32'
    PRETRAINED = 'openai'
    
    # Which layers to extract activations from
    # SAEs are typically applied to residual stream or MLP outputs
    VISION_LAYERS_TO_HOOK = ['resblocks.0', 'resblocks.5', 'resblocks.11']  # early, mid, late
    TEXT_LAYERS_TO_HOOK = ['resblocks.0', 'resblocks.5', 'resblocks.11']
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Sample data for verification
    SAMPLE_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
    SAMPLE_TEXTS = [
        "a photo of a cat",
        "a photo of a dog", 
        "a beautiful landscape",
        "a person walking"
    ]

config = Config()
print(f"Device: {config.DEVICE}")
print(f"Model: {config.MODEL_NAME} (pretrained: {config.PRETRAINED})")

Device: cuda
Model: ViT-B-32 (pretrained: openai)


In [3]:
def load_clip_model(model_name, pretrained, device):
    """
    Load a CLIP model using open_clip.
    
    Returns:
        model: The CLIP model
        preprocess: Image preprocessing transform
        tokenizer: Text tokenizer
    """
    print(f"\n{'='*60}")
    print(f"Loading CLIP model: {model_name} ({pretrained})")
    print(f"{'='*60}")
    
    model, _, preprocess = create_model_and_transforms(
        model_name, 
        pretrained=pretrained,
        device=device
    )
    tokenizer = get_tokenizer(model_name)
    
    model.eval()  # Set to evaluation mode — important for consistent activations
    
    # Print model architecture summary
    print(f"\n Model Architecture Summary ")
    
    # Vision encoder info
    vision = model.visual
    print(f"\nVision Encoder:")
    print(f"  Type: {type(vision).__name__}")
    if hasattr(vision, 'transformer'):
        num_vision_layers = len(vision.transformer.resblocks)
        print(f"  Number of transformer layers: {num_vision_layers}")
    if hasattr(vision, 'conv1'):
        print(f"  Patch embedding conv: {vision.conv1}")
    if hasattr(vision, 'proj'):
        if vision.proj is not None:
            print(f"  Projection dim: {vision.proj.shape}")
    
    # Text encoder info
    transformer = model.transformer
    print(f"\nText Encoder:")
    num_text_layers = len(transformer.resblocks)
    print(f"  Number of transformer layers: {num_text_layers}")
    if hasattr(model, 'text_projection'):
        print(f"  Text projection shape: {model.text_projection.shape}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    vision_params = sum(p.numel() for p in model.visual.parameters())
    text_params = total_params - vision_params
    print(f"\nParameter counts:")
    print(f"  Total: {total_params:,}")
    print(f"  Vision: {vision_params:,}")
    print(f"  Text: {text_params:,}")
    
    return model, preprocess, tokenizer


model, preprocess, tokenizer = load_clip_model(
    config.MODEL_NAME, config.PRETRAINED, config.DEVICE
)


Loading CLIP model: ViT-B-32 (openai)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



--- Model Architecture Summary ---

Vision Encoder:
  Type: VisionTransformer
  Number of transformer layers: 12
  Patch embedding conv: Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
  Projection dim: torch.Size([768, 512])

Text Encoder:
  Number of transformer layers: 12
  Text projection shape: torch.Size([512, 512])

Parameter counts:
  Total: 151,277,313
  Vision: 87,849,216
  Text: 63,428,097


In [4]:
# Activation Extraction Hook System
class ActivationStore:
    """
    Hooks into CLIP layers to capture intermediate activations.
    
    These activations are what SAEs will be trained on — the SAE learns
    to decompose dense activation vectors into sparse, interpretable features.
    
    Based on methodology from:
    - "Sparse Autoencoders Learn Monosemantic Features in VLMs" (SAEs_in_vision_models.pdf)
    - "Steering in CLIP" (Steering_in_CLIP.pdf)
    """
    
    def __init__(self):
        self.activations = OrderedDict()
        self.hooks = []
    
    def _get_hook(self, name):
        """Create a forward hook that stores the output of a layer."""
        def hook_fn(module, input, output):
            # Handle different output types
            if isinstance(output, tuple):
                self.activations[name] = output[0].detach().cpu()
            else:
                self.activations[name] = output.detach().cpu()
        return hook_fn
    
    def register_hooks_on_vision(self, model, layer_names):
        """
        Register hooks on vision encoder layers.
        
        For CLIP ViT, the residual stream activations at each transformer
        block are the primary targets for SAE analysis.
        """
        vision_transformer = model.visual.transformer
        
        for name in layer_names:
            parts = name.split('.')
            module = vision_transformer
            for part in parts:
                if part.isdigit():
                    module = module[int(part)]
                else:
                    module = getattr(module, part)
            
            hook = module.register_forward_hook(self._get_hook(f"vision.{name}"))
            self.hooks.append(hook)
            print(f"  Hooked: vision.{name} -> {type(module).__name__}")
    
    def register_hooks_on_text(self, model, layer_names):
        """Register hooks on text encoder layers."""
        text_transformer = model.transformer
        
        for name in layer_names:
            parts = name.split('.')
            module = text_transformer
            for part in parts:
                if part.isdigit():
                    module = module[int(part)]
                else:
                    module = getattr(module, part)
            
            hook = module.register_forward_hook(self._get_hook(f"text.{name}"))
            self.hooks.append(hook)
            print(f"  Hooked: text.{name} -> {type(module).__name__}")
    
    def clear(self):
        """Clear stored activations."""
        self.activations.clear()
    
    def remove_hooks(self):
        """Remove all hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()
    
    def get_activation_shapes(self):
        """Return shapes of all stored activations."""
        return {name: act.shape for name, act in self.activations.items()}


# Set up the activation store
print(f"\n{'='*60}")
print("Setting up activation extraction hooks")
print(f"{'='*60}")

activation_store = ActivationStore()
activation_store.register_hooks_on_vision(model, config.VISION_LAYERS_TO_HOOK)
activation_store.register_hooks_on_text(model, config.TEXT_LAYERS_TO_HOOK)


Setting up activation extraction hooks
  Hooked: vision.resblocks.0 -> ResidualAttentionBlock
  Hooked: vision.resblocks.5 -> ResidualAttentionBlock
  Hooked: vision.resblocks.11 -> ResidualAttentionBlock
  Hooked: text.resblocks.0 -> ResidualAttentionBlock
  Hooked: text.resblocks.5 -> ResidualAttentionBlock
  Hooked: text.resblocks.11 -> ResidualAttentionBlock


In [5]:
# Verify with Sample Inputs
def load_sample_image(url):
    """Download and load a sample image."""
    try:
        response = requests.get(url, timeout=10)
        image = Image.open(BytesIO(response.content)).convert('RGB')
        print(f"  Loaded image: {image.size}")
        return image
    except Exception as e:
        print(f"  Failed to load from URL: {e}")
        print("  Creating synthetic test image...")
        image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        return image


def verify_model(model, preprocess, tokenizer, activation_store, config):
    """
    Run a forward pass with sample data to verify:
    1. Model loads correctly
    2. Hooks capture activations
    3. Shapes are as expected for SAE training
    """
    print(f"\n{'='*60}")
    print("Verifying model with sample inputs")
    print(f"{'='*60}")
    
    # Load sample image
    print("\n Loading sample image ")
    image = load_sample_image(config.SAMPLE_IMAGE_URL)
    image_input = preprocess(image).unsqueeze(0).to(config.DEVICE)
    print(f"  Preprocessed image tensor: {image_input.shape}")
    
    # Tokenize sample texts
    print("\n Tokenizing sample texts ")
    text_input = tokenizer(config.SAMPLE_TEXTS).to(config.DEVICE)
    print(f"  Tokenized text tensor: {text_input.shape}")
    
    # Forward pass
    print("\n Running forward pass ")
    activation_store.clear()
    
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)
    
    # Normalize features (CLIP uses cosine similarity)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    
    print(f"  Image features shape: {image_features.shape}")
    print(f"  Text features shape: {text_features.shape}")
    
    # Compute similarities
    similarity = (image_features @ text_features.T).cpu().numpy()
    print(f"\n Image-Text Similarities ")
    for i, text in enumerate(config.SAMPLE_TEXTS):
        print(f"  '{text}': {similarity[0][i]:.4f}")
    
    # Check captured activations
    print(f"\n Captured Activations ")
    shapes = activation_store.get_activation_shapes()
    for name, shape in shapes.items():
        print(f"  {name}: {shape}")
    
    return image_features, text_features, shapes


image_features, text_features, activation_shapes = verify_model(
    model, preprocess, tokenizer, activation_store, config
)


Verifying model with sample inputs

 Loading sample image 
  Failed to load from URL: cannot identify image file <_io.BytesIO object at 0x7eaa68a058f0>
  Creating synthetic test image...
  Preprocessed image tensor: torch.Size([1, 3, 224, 224])

 Tokenizing sample texts 
  Tokenized text tensor: torch.Size([4, 77])

 Running forward pass 
  Image features shape: torch.Size([1, 512])
  Text features shape: torch.Size([4, 512])

 Image-Text Similarities 
  'a photo of a cat': 0.2309
  'a photo of a dog': 0.2260
  'a beautiful landscape': 0.2254
  'a person walking': 0.2166

 Captured Activations 
  vision.resblocks.0: torch.Size([1, 50, 768])
  vision.resblocks.5: torch.Size([1, 50, 768])
  vision.resblocks.11: torch.Size([1, 50, 768])
  text.resblocks.0: torch.Size([4, 77, 512])
  text.resblocks.5: torch.Size([4, 77, 512])
  text.resblocks.11: torch.Size([4, 77, 512])


In [6]:
# Analyze Activation Properties

def analyze_activations(activation_store):
    """
    Analyze activation statistics — this informs SAE architecture decisions.
    
    Key things to know for SAE training:
    - Activation dimension (d_model) → SAE input/output dim
    - Activation statistics → normalization strategy
    - Sparsity patterns → SAE dictionary size choice
    """
    print(f"\n{'='*60}")
    print("Activation Analysis for SAE Planning")
    print(f"{'='*60}")
    
    results = {}
    
    for name, activation in activation_store.activations.items():
        print(f"\n {name} ")
        
        # Reshape: [batch, seq_len, d_model] or [batch, d_model]
        if activation.dim() == 3:
            batch, seq_len, d_model = activation.shape
            flat = activation.reshape(-1, d_model)
            print(f"  Shape: [{batch}, {seq_len}, {d_model}]")
            print(f"  Sequence length: {seq_len} (vision: patches+CLS, text: tokens)")
        elif activation.dim() == 2:
            batch, d_model = activation.shape
            flat = activation
            seq_len = 1
            print(f"  Shape: [{batch}, {d_model}]")
        else:
            print(f"  Unexpected shape: {activation.shape}")
            continue
        
        # Statistics
        mean = flat.mean().item()
        std = flat.std().item()
        min_val = flat.min().item()
        max_val = flat.max().item()
        
        # Sparsity (fraction of near-zero activations)
        sparsity = (flat.abs() < 0.01).float().mean().item()
        
        print(f"  d_model (SAE input dim): {d_model}")
        print(f"  Mean: {mean:.6f}")
        print(f"  Std:  {std:.6f}")
        print(f"  Range: [{min_val:.4f}, {max_val:.4f}]")
        print(f"  Near-zero sparsity (<0.01): {sparsity:.4f}")
        
        # Recommended SAE dictionary size
        # Common practice: 4x to 64x expansion (from SAEs_in_vision_models.pdf)
        for expansion in [4, 8, 16, 32]:
            dict_size = d_model * expansion
            print(f"  SAE dict size ({expansion}x): {dict_size}")
        
        results[name] = {
            'd_model': d_model,
            'seq_len': seq_len,
            'mean': mean,
            'std': std,
            'sparsity': sparsity,
        }
    
    return results


activation_analysis = analyze_activations(activation_store)


Activation Analysis for SAE Planning

 vision.resblocks.0 
  Shape: [1, 50, 768]
  Sequence length: 50 (vision: patches+CLS, text: tokens)
  d_model (SAE input dim): 768
  Mean: 0.005169
  Std:  0.345999
  Range: [-5.6320, 2.6918]
  Near-zero sparsity (<0.01): 0.0328
  SAE dict size (4x): 3072
  SAE dict size (8x): 6144
  SAE dict size (16x): 12288
  SAE dict size (32x): 24576

 vision.resblocks.5 
  Shape: [1, 50, 768]
  Sequence length: 50 (vision: patches+CLS, text: tokens)
  d_model (SAE input dim): 768
  Mean: 0.022948
  Std:  0.143018
  Range: [-4.7981, 0.9510]
  Near-zero sparsity (<0.01): 0.0661
  SAE dict size (4x): 3072
  SAE dict size (8x): 6144
  SAE dict size (16x): 12288
  SAE dict size (32x): 24576

 vision.resblocks.11 
  Shape: [1, 50, 768]
  Sequence length: 50 (vision: patches+CLS, text: tokens)
  d_model (SAE input dim): 768
  Mean: 0.019128
  Std:  0.534224
  Range: [-10.2171, 9.2619]
  Near-zero sparsity (<0.01): 0.0236
  SAE dict size (4x): 3072
  SAE dict size 

In [7]:
# Define SAE Architecture (ready for Task T-02 training)

class SparseAutoencoder(nn.Module):
    """
    Sparse Autoencoder for decomposing CLIP activations into 
    monosemantic features.
    
    Architecture from "Sparse Autoencoders Learn Monosemantic Features 
    in Vision-Language Models" (SAEs_in_vision_models.pdf):
    
    Encoder: h = ReLU(W_enc @ (x - b_dec) + b_enc)
    Decoder: x_hat = W_dec @ h + b_dec
    
    Where:
    - x: input activation vector (d_model)
    - h: sparse feature vector (d_sae), most entries ~0
    - W_enc: [d_sae, d_model] encoder weights
    - W_dec: [d_model, d_sae] decoder weights (dictionary)
    - b_enc: [d_sae] encoder bias
    - b_dec: [d_model] decoder bias (pre-encoder subtracted)
    """
    
    def __init__(self, d_model, d_sae, dtype=torch.float32):
        super().__init__()
        
        self.d_model = d_model
        self.d_sae = d_sae
        
        # Encoder
        self.W_enc = nn.Parameter(torch.empty(d_sae, d_model, dtype=dtype))
        self.b_enc = nn.Parameter(torch.zeros(d_sae, dtype=dtype))
        
        # Decoder
        self.W_dec = nn.Parameter(torch.empty(d_model, d_sae, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_model, dtype=dtype))
        
        # Initialize
        self._init_weights()
    
    def _init_weights(self):
        """Kaiming initialization for encoder, normalize decoder columns."""
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        
        # Normalize decoder columns (dictionary elements should be unit norm)
        with torch.no_grad():
            self.W_dec.data = self.W_dec.data / self.W_dec.data.norm(dim=0, keepdim=True)
    
    def encode(self, x):
        """Encode input to sparse features."""
        return torch.relu(
            (x - self.b_dec) @ self.W_enc.T + self.b_enc
        )
    
    def decode(self, h):
        """Decode sparse features back to activation space."""
        return h @ self.W_dec.T + self.b_dec
    
    def forward(self, x):
        """
        Full forward pass.
        
        Returns:
            x_hat: reconstructed activation
            h: sparse feature activations
            loss_dict: reconstruction loss + sparsity loss components
        """
        h = self.encode(x)          # [batch, d_sae] — sparse
        x_hat = self.decode(h)       # [batch, d_model] — reconstruction
        
        # Reconstruction loss (MSE)
        recon_loss = (x - x_hat).pow(2).sum(dim=-1).mean()
        
        # Sparsity loss (L1 on features)
        sparsity_loss = h.abs().sum(dim=-1).mean()
        
        return x_hat, h, {
            'recon_loss': recon_loss,
            'sparsity_loss': sparsity_loss,
        }


# Create an SAE instance for verification
print(f"\n{'='*60}")
print("Initializing SAE (architecture verification)")
print(f"{'='*60}")

# Use d_model from the last vision layer
sample_layer = list(activation_analysis.keys())[0]
d_model = activation_analysis[sample_layer]['d_model']
expansion_factor = 8  # 8x expansion is a good starting point
d_sae = d_model * expansion_factor

sae = SparseAutoencoder(d_model=d_model, d_sae=d_sae).to(config.DEVICE)

print(f"  d_model (input): {d_model}")
print(f"  d_sae (features): {d_sae}")
print(f"  Expansion factor: {expansion_factor}x")
print(f"  SAE parameters: {sum(p.numel() for p in sae.parameters()):,}")

# Quick test with captured activations
sample_act = list(activation_store.activations.values())[0]
if sample_act.dim() == 3:
    test_input = sample_act[:, 0, :].to(config.DEVICE)  # Take CLS token
else:
    test_input = sample_act.to(config.DEVICE)

with torch.no_grad():
    x_hat, h, losses = sae(test_input)

print(f"\n  Test forward pass:")
print(f"    Input shape: {test_input.shape}")
print(f"    Reconstruction shape: {x_hat.shape}")
print(f"    Feature shape: {h.shape}")
print(f"    Active features (>0): {(h > 0).sum().item()} / {h.numel()}")
print(f"    Feature sparsity: {(h == 0).float().mean().item():.4f}")
print(f"    Reconstruction loss: {losses['recon_loss'].item():.4f}")
print(f"    Sparsity loss: {losses['sparsity_loss'].item():.4f}")


Initializing SAE (architecture verification)
  d_model (input): 768
  d_sae (features): 6144
  Expansion factor: 8x
  SAE parameters: 9,444,096

  Test forward pass:
    Input shape: torch.Size([1, 768])
    Reconstruction shape: torch.Size([1, 768])
    Feature shape: torch.Size([1, 6144])
    Active features (>0): 3070 / 6144
    Feature sparsity: 0.5003
    Reconstruction loss: 369.4539
    Sparsity loss: 841.4626


In [8]:
# Summary & Save Configuration

print(f"\n{'='*60}")
print("T-01 SUMMARY: Model Loading Complete")
print(f"{'='*60}")

summary = {
    'clip_model': config.MODEL_NAME,
    'pretrained': config.PRETRAINED,
    'device': config.DEVICE,
    'vision_layers_hooked': config.VISION_LAYERS_TO_HOOK,
    'text_layers_hooked': config.TEXT_LAYERS_TO_HOOK,
    'activation_shapes': activation_shapes,
    'activation_analysis': activation_analysis,
    'sae_config': {
        'd_model': d_model,
        'd_sae': d_sae,
        'expansion_factor': expansion_factor,
    }
}

for key, val in summary.items():
    if isinstance(val, dict):
        print(f"\n  {key}:")
        for k2, v2 in val.items():
            print(f"    {k2}: {v2}")
    else:
        print(f"  {key}: {val}")

# Save for downstream tasks
torch.save({
    'config': summary,
    'sae_state_dict': sae.state_dict(),
}, 't01_checkpoint.pt')

print(f"\n  Checkpoint saved to: t01_checkpoint.pt")
print(f"\n  Ready for T-02: SAE Training on CLIP activations")

# Cleanup hooks (re-register when needed)
activation_store.remove_hooks()
print("  Hooks removed (re-register for data collection)")


T-01 SUMMARY: Model Loading Complete
  clip_model: ViT-B-32
  pretrained: openai
  device: cuda
  vision_layers_hooked: ['resblocks.0', 'resblocks.5', 'resblocks.11']
  text_layers_hooked: ['resblocks.0', 'resblocks.5', 'resblocks.11']

  activation_shapes:
    vision.resblocks.0: torch.Size([1, 50, 768])
    vision.resblocks.5: torch.Size([1, 50, 768])
    vision.resblocks.11: torch.Size([1, 50, 768])
    text.resblocks.0: torch.Size([4, 77, 512])
    text.resblocks.5: torch.Size([4, 77, 512])
    text.resblocks.11: torch.Size([4, 77, 512])

  activation_analysis:
    vision.resblocks.0: {'d_model': 768, 'seq_len': 50, 'mean': 0.0051694950088858604, 'std': 0.3459988832473755, 'sparsity': 0.03283854201436043}
    vision.resblocks.5: {'d_model': 768, 'seq_len': 50, 'mean': 0.02294803224503994, 'std': 0.14301803708076477, 'sparsity': 0.06606771051883698}
    vision.resblocks.11: {'d_model': 768, 'seq_len': 50, 'mean': 0.01912766508758068, 'std': 0.5342237949371338, 'sparsity': 0.0236197